# Checkup figures

Makes the figures for the Nov 4 checkup deck from the specs in [`docs/CHECKUP-VISUALS.md`](../docs/CHECKUP-VISUALS.md). Every figure is saved under `docs/figures/` with the cell that made it.

| Cell | Figure | Deck slot | Needs the data |
|---|---|---|---|
| A | per-class IoU, ours beside the paper, with the gap | slide 7 | no |
| B | architecture strip, current vs goal | slide 8 | no |
| C | one cube: false colour, mask, spectrum | slide 6 or 3 | yes |
| D | mean spectrum per class, 8 richest cubes each | slide 3 or 6 | yes |
| G | training curves from `metrics.csv` | Nov 5 | a metrics file |

Numbers in A come from `docs/RESULTS.md` (row `cnn_ds2`) and Casao et al., IROS 2024, Table IV. Shapes in B come from `src/sw/models/transformer.py`. Cells C and D read `labels_hyper_lt/train` and `hyper/train` under `SW_ROOT` and skip themselves if it is not set. D takes the 8 train cubes with the most pixels of each object class, so basket, tape, and filament are on the chart.

Class colours are the dataset palette from `meta.json` (`sw.data.PALETTE`), so the charts agree with the mask. Light colours (film, basket, tape, bag) are drawn with a dark halo, and every bar has an ink edge and a printed value.


## Colab setup

Skip on your own machine. On Colab it clones the repo and installs the libraries. The 23 GB download runs only when `DOWNLOAD_DATA` is True; cells A, B and G do not need it.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DOWNLOAD_DATA = False  # True only for cells C and D (about 23 GB)
if IN_COLAB:
    os.chdir("/content")
    repo = Path("/content/255-final-project")
    if not repo.is_dir():
        subprocess.check_call(
            ["git", "clone", "https://github.com/scottn66/255-final-project.git", str(repo)]
        )
    os.chdir(repo)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    if DOWNLOAD_DATA and not os.environ.get("SW_ROOT"):
        subprocess.check_call(["bash", "scripts/get_data.sh", "/content/data"])
        os.environ["SW_ROOT"] = "/content/data/spectralwaste_segmentation"

## Setup

Imports, the class palette, the output folder, and the nominal wavelength axis. `meta.json` ships no wavelengths, so the nm axis assumes even spacing of the 224 bands over 900 to 1700 nm and says so.

In [ ]:
import os
import sys
from pathlib import Path


def _find_repo_root():
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "src" / "sw" / "__init__.py").is_file():
            return cand
    raise RuntimeError("Start Jupyter from the repo root or from the notebooks/ folder.")


REPO = _find_repo_root()
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyBboxPatch, Patch

from sw.data import CLASS_NAMES, HYPER_LT_TRAIN_PIXEL_COUNTS, PALETTE, UINT16_MAX

FIG_DIR = REPO / "docs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SW_ROOT = os.environ.get("SW_ROOT")

OBJECT_CLASSES = CLASS_NAMES[1:]  # film .. bag, ids 1..6
COLOR = {name: np.array(rgb) / 255 for name, rgb in zip(CLASS_NAMES, PALETTE)}
LABEL = {**{name: name for name in CLASS_NAMES}, "video_tape": "tape"}
INK, MUTED, PAPER_FACE = "#222222", "#5c5c5c", "#f3f3f3"
PER_CLASS = 8  # richest train cubes per object class, cell D

N_BANDS, GROUP, N_GROUPS = 224, 16, 14
FALSE_COLOUR_BANDS = (56, 112, 168)  # notebook 01: C // 4, C // 2, 3C // 4
LINESTYLE = {
    "background": ":",
    "film": "-",
    "basket": "--",
    "cardboard": "-",
    "video_tape": "-.",
    "filament": "-",
    "bag": (0, (5, 1.6)),
}


def nominal_nm(band):
    """Nominal wavelength. meta.json ships none; assume even spacing over 900-1700 nm."""
    return 900 + np.asarray(band, dtype=float) * 800 / (N_BANDS - 1)


def band_of_nm(nm):
    return (np.asarray(nm, dtype=float) - 900) * (N_BANDS - 1) / 800


plt.rcParams.update(
    {
        "font.size": 12,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 12,
        "legend.fontsize": 10,
        "axes.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "hatch.linewidth": 0.6,
        "savefig.dpi": 200,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.28,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
    }
)


def save(fig, name, svg=False):
    """Write docs/figures/<name>.png (and .svg when asked) and print the path."""
    png = FIG_DIR / f"{name}.png"
    fig.savefig(png)
    if svg:
        fig.savefig(FIG_DIR / f"{name}.svg")
    print("saved", png.relative_to(REPO))
    plt.close(fig)
    return png


def train_dirs():
    """Label and cube folders for the train split. Labels may outnumber cubes."""
    root = Path(SW_ROOT)
    labels = root / "labels_hyper_lt" / "train"
    cubes = root / "hyper" / "train"
    if not labels.is_dir():
        raise FileNotFoundError(f"missing {labels}")
    return labels, cubes


def load_mask(path):
    from PIL import Image

    with Image.open(path) as img:
        mask = np.array(img)
    if mask.ndim != 2:
        raise ValueError(f"{path.name} has shape {mask.shape}; expected (H, W) class ids")
    return mask


def load_cube(path):
    import tifffile

    cube = np.asarray(tifffile.imread(path), dtype=np.float32) / np.float32(UINT16_MAX)
    if cube.ndim != 3 or cube.shape[0] != N_BANDS:
        raise ValueError(f"{path.name} has shape {cube.shape}; expected ({N_BANDS}, H, W)")
    return cube


def class_counts(mask):
    counts = np.zeros(len(CLASS_NAMES), dtype=np.int64)
    ids, n = np.unique(mask, return_counts=True)
    for i, k in zip(ids, n):
        if 0 <= int(i) < len(counts):
            counts[int(i)] = int(k)
    return counts


def choose_example_stem(label_dir):
    """Filament present, then the most classes, then the earliest stem."""
    best = None
    for index, path in enumerate(sorted(label_dir.glob("*.png"))):
        ids = np.unique(load_mask(path))
        key = (5 in set(ids.tolist()), int(len(ids)), -index)
        if best is None or key > best[0]:
            best = (key, path.stem)
    return best[1]


def choose_class_cubes(label_dir, per_class=PER_CLASS):
    """Up to `per_class` stems with the most pixels of each object class."""
    stems = sorted(p.stem for p in label_dir.glob("*.png"))
    counts = np.zeros((len(stems), len(CLASS_NAMES)), dtype=np.int64)
    for i, stem in enumerate(stems):
        counts[i] = class_counts(load_mask(label_dir / f"{stem}.png"))
    chosen = {}
    for c, name in enumerate(CLASS_NAMES):
        if name == "background":
            continue
        order = np.argsort(-counts[:, c], kind="stable")
        picked = []
        for i in order:
            if counts[i, c] <= 0:
                break
            picked.append(stems[int(i)])
            if len(picked) >= per_class:
                break
        chosen[name] = picked
    return chosen


def draw_line(ax, x, y, name, label=None, lw=2.3):
    """Palette colour with a dark halo so film, basket, tape, and bag stay visible."""
    color = "0.45" if name == "background" else COLOR[name]
    ls = LINESTYLE[name]
    if name == "background":
        ax.plot(x, y, color=color, lw=1.7, ls=ls, label=label, zorder=3)
        return
    ax.plot(x, y, color=INK, lw=lw + 1.35, ls=ls, solid_capstyle="round", zorder=3)
    ax.plot(x, y, color=color, lw=lw, ls=ls, solid_capstyle="round", label=label, zorder=4)


print("repo", REPO)
print("SW_ROOT", SW_ROOT or "(not set: cells C and D will skip)")


## A. Per-class IoU (slide 7)

Ours is val, half resolution for cube and mask, 30 epochs, best epoch 29, one run. The gray bars are the paper's MiniNet-v2 HYPER row, the fair comparison because MiniNetSeg is from the same family. Background (86.8) stays off the bars: it is not in the foreground mean. SegFormer-B0 HYPER 54.3 is named in the footer as the other published hyperspectral number, not drawn as a third bar.

Classes are sorted by the gap (paper minus ours). The right-hand labels are that gap in points.


In [ ]:
# docs/RESULTS.md, row cnn_ds2, best val FG mIoU 0.391 at epoch 29, in percent
OURS = {"film": 55.2, "basket": 49.5, "cardboard": 57.8, "video_tape": 28.6, "filament": 0.0, "bag": 43.6}
OURS_MEAN = 39.1
# Casao et al., IROS 2024, Table IV, MiniNet-v2 HYPER row, in percent
PAPER = {"film": 61.2, "basket": 61.0, "cardboard": 78.8, "video_tape": 28.8, "filament": 30.5, "bag": 56.3}
PAPER_MEAN = 52.8

assert abs(np.mean(list(OURS.values())) - OURS_MEAN) < 0.05
assert abs(np.mean(list(PAPER.values())) - PAPER_MEAN) < 0.05

gap = {c: PAPER[c] - OURS[c] for c in OBJECT_CLASSES}
order = sorted(OBJECT_CLASSES, key=lambda c: gap[c], reverse=True)
gap_share = (gap["filament"] + gap["cardboard"]) / sum(gap.values())
y = np.arange(len(order))[::-1]
h = 0.36

fig, ax = plt.subplots(figsize=(12.2, 6.05))
ax.barh(
    y + h / 2,
    [OURS[c] for c in order],
    height=h,
    color=[COLOR[c] for c in order],
    edgecolor=INK,
    linewidth=0.8,
    zorder=2,
)
ax.barh(
    y - h / 2,
    [PAPER[c] for c in order],
    height=h,
    color=PAPER_FACE,
    edgecolor=INK,
    linewidth=0.8,
    zorder=2,
)
for yi, c in zip(y, order):
    if OURS[c] == 0:
        # A zero-width bar disappears, so mark the class colour at the axis.
        ax.plot([1.1, 1.1], [yi + 0.04, yi + h - 0.04], color=COLOR[c], lw=6, solid_capstyle="butt", zorder=4)
        ax.text(2.6, yi + h / 2, "0.0", va="center", ha="left", fontsize=11, color=INK, fontweight="bold", zorder=4)
    else:
        ax.text(OURS[c] + 1.1, yi + h / 2, f"{OURS[c]:.1f}", va="center", ha="left", fontsize=11, color=INK, zorder=4)
    ax.text(PAPER[c] + 1.1, yi - h / 2, f"{PAPER[c]:.1f}", va="center", ha="left", fontsize=11, color=MUTED, zorder=4)

ax.set_yticks(y, [LABEL[c] for c in order])
ax.set_xlim(0, 100)
ax.set_ylim(-0.72, y[0] + 0.85)
ax.set_xlabel("IoU (%)")
ax.grid(axis="x", color="#ececec", zorder=0)
ax.set_axisbelow(True)

ax_r = ax.twinx()
ax_r.set_ylim(ax.get_ylim())
ax_r.set_yticks(y, [f"{gap[c]:.1f}" for c in order])
ax_r.set_ylabel("gap, paper minus ours (points)")
ax_r.tick_params(axis="y", length=0, labelsize=12, colors=INK)
for tick, c in zip(ax_r.get_yticklabels(), order):
    tick.set_fontweight("bold" if gap[c] >= 20 else "regular")
ax_r.spines["top"].set_visible(False)
ax_r.spines["left"].set_visible(False)
ax_r.spines["right"].set_color("#bdbdbd")

ax.text(
    0.985,
    0.02,
    "upper bar · ours, in that class colour\nlower bar · paper, MiniNet-v2 HYPER",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=10.5,
    color=INK,
    linespacing=1.45,
)

claim = f"Foreground mIoU {OURS_MEAN} vs paper {PAPER_MEAN}. Filament and cardboard are {gap_share:.0%} of the gap."
ax.set_title("Per-class IoU, first CNN run\n" + claim, loc="left", color=INK, pad=10)
fig.text(
    0.0,
    -0.02,
    "Ours: MiniNetSeg, 571,879 params; half resolution (cube and mask); 30 epochs; val; best epoch 29; one run. "
    "Background IoU 86.8 is not in the foreground mean.",
    fontsize=9,
    color=MUTED,
)
fig.text(
    0.0,
    -0.07,
    "Paper bars: Casao et al., IROS 2024, Table IV, MiniNet-v2 HYPER. "
    "SegFormer-B0 HYPER on that table is 54.3 (full res, 200 epochs), context rather than a third bar.",
    fontsize=9,
    color=MUTED,
)
fig.tight_layout()
save(fig, "per_class_iou")
print("gap (points):", {LABEL[c]: round(gap[c], 1) for c in order})
print("filament + cardboard share of the gap:", round(gap_share, 2))


## B. Architecture strip, current vs goal (slide 8)

Shapes per stage at a 256 input, from `src/sw/models/transformer.py`. Spectral block and spectral pool landed in PR #12 (`ResidualSelfAttentionBlock` across the 14 band groups, then a mean over groups). Spatial PE, the spatial blocks, and the seg head still raise `NotImplementedError`.

Status is box style plus a word: solid is merged or done, dashed outline is a stub. The editable version for the slide is drawn in Slides from this same table; this cell is the fallback and writes an SVG too.


In [ ]:
STAGES = [
    # title, detail, output shape, params, status
    ("Input cube", "loader: uint16 scaled to [0, 1]", "(B, 224, 256, 256)", "0", "done"),
    (
        "Band tokenizer",
        "14 groups x 16 bands\nshared Conv2d 16 to 64, k8 s8\nLayerNorm + group emb. 14 x 64",
        "(B, 1024, 14, 64)",
        "66,624",
        "merged, PR #9",
    ),
    (
        "Spectral block x2",
        "attention over the 14 band\ngroups, inside each patch",
        "(B, 1024, 14, 64)",
        "49,984 each",
        "merged, PR #12",
    ),
    ("Spectral pool", "mean over the 14 groups", "(B, 1024, 64)", "0", "merged, PR #12"),
    ("Spatial PE", "fixed 2D sin-cos\non the 32 x 32 grid", "(B, 1024, 64)", "0", "stub"),
    ("Spatial block x6", "attention over the\n1,024 patches", "(B, 1024, 64)", "49,984 each", "stub"),
    ("Seg head", "(B, 64, 32, 32) to 7 classes\nupsample x8", "(B, 7, 256, 256)", "~0.5k (1x1) to ~21k", "stub"),
    (
        "Loss + metric",
        "CE with paper-scheme weights\non the HSI counts;\nFG mIoU over classes 1 to 6",
        "scalar",
        "0",
        "done (harness)",
    ),
]
PER_BLOCK, N_BLOCKS, TOKENIZER = 49_984, 8, 66_624
assert TOKENIZER + N_BLOCKS * PER_BLOCK == 466_496


def kind_of(status):
    if status.startswith("stub"):
        return "stub"
    if status.startswith("in review"):
        return "review"
    return "done"


def draw_box(x, y, w, h, status, rounding):
    """Solid = merged or done, hatched = in review, dashed outline = stub."""
    kind = kind_of(status)
    style = f"round,pad=0,rounding_size={rounding}"
    if kind == "review":
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=style, facecolor="white", edgecolor="#a6a6a6", hatch="///", linewidth=0))
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=style, facecolor="none", edgecolor=INK, linewidth=1.4))
    elif kind == "stub":
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=style, facecolor="white", edgecolor=INK, linewidth=1.5, linestyle=(0, (4, 3))))
    else:
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=style, facecolor="#ececec", edgecolor=INK, linewidth=1.4))


def text_bg(status):
    """A pad behind box text so a hatch never runs through the words."""
    return dict(facecolor="#ececec" if kind_of(status) == "done" else "white", edgecolor="none", pad=1.6)


W, H = 0.88, 0.70
ROW_Y = {0: 1.22, 1: 0.28}
fig, ax = plt.subplots(figsize=(13.6, 8.15))
ax.set_xlim(0, 4)
ax.set_ylim(0, 2.34)
ax.axis("off")
ax.text(0.04, 2.28, "Band-token transformer, spectral then spatial", fontsize=16, fontweight="bold", va="top", color=INK)
ax.text(
    0.04,
    2.14,
    "Merged through the spectral pool. The three dashed stages are still stubs, so there is no transformer score.",
    fontsize=11,
    va="top",
    color=MUTED,
)

for i, (title, detail, shape, params, status) in enumerate(STAGES):
    row, col = divmod(i, 4)
    x0, y0 = 0.05 + col * 0.99, ROW_Y[row]
    draw_box(x0, y0, W, H, status, 0.03)
    cx, bg = x0 + W / 2, text_bg(status)
    ax.text(cx, y0 + H - 0.045, title, ha="center", va="top", fontsize=12, fontweight="bold", color=INK, bbox=bg)
    ax.text(cx, y0 + H - 0.155, detail, ha="center", va="top", fontsize=9, color=INK, linespacing=1.25, bbox=bg)
    ax.text(cx, y0 + 0.20, shape, ha="center", va="center", fontsize=9, family="monospace", color=INK, bbox=bg)
    ax.text(cx, y0 + 0.115, f"params {params}", ha="center", va="center", fontsize=9, color=MUTED, bbox=bg)
    ax.text(cx, y0 + 0.04, status, ha="center", va="center", fontsize=9.5, style="italic", color=INK, bbox=bg)
    if col < 3:
        yc = y0 + 0.30
        ax.annotate("", xy=(x0 + 0.99, yc), xytext=(x0 + W, yc), arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.2))

top_end = (0.05 + 3 * 0.99 + W / 2, ROW_Y[0])
mid_y = ROW_Y[0] - 0.10
start2 = (0.05 + W / 2, ROW_Y[1] + H)
ax.plot([top_end[0], top_end[0], start2[0]], [top_end[1], mid_y, mid_y], color=INK, lw=1.2)
ax.annotate("", xy=start2, xytext=(start2[0], mid_y), arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.2))

legend_items = [("merged or done", "done"), ("stub, still to do", "stub")]
for k, (word, status) in enumerate(legend_items):
    draw_box(0.05 + 1.15 * k, 0.145, 0.16, 0.075, status, 0.015)
    ax.text(0.05 + 1.15 * k + 0.20, 0.182, word, va="center", fontsize=11, color=INK)
ax.text(
    0.05,
    0.08,
    f"Designed, not yet measured: {TOKENIZER:,} + {N_BLOCKS} x {PER_BLOCK:,} = {TOKENIZER + N_BLOCKS * PER_BLOCK:,} + head, about 0.47M (1x1 head) to 0.49M (recipe head).",
    va="center",
    fontsize=10,
    color=INK,
)
ax.text(
    0.05,
    0.025,
    "Half resolution (--downsample 2): 16 x 16 grid, 256 tokens.   Next: spatial PE, spatial blocks, seg head, then the Phase 2 ablation.",
    va="center",
    fontsize=10,
    color=MUTED,
)
save(fig, "architecture", svg=True)


## C. One cube: false colour, mask, spectrum (slide 6 or 3)

Needs `SW_ROOT`. Scans every train mask (no cube reads) and keeps the earliest cube that contains filament and the most classes. The stem is printed on the figure. The false-colour panel is three infrared bands, contrast-stretched for display. The spectrum uses the same scaled values as the loader, with no stretch, and a legend instead of labels at the right edge.


In [ ]:
if not SW_ROOT:
    print("SW_ROOT is not set; skipping cell C.")
else:
    label_dir, cube_dir = train_dirs()
    stem = choose_example_stem(label_dir)
    cube_path = cube_dir / f"{stem}.tiff"
    if not cube_path.is_file():
        raise FileNotFoundError(f"missing {cube_path}")
    cube_np = load_cube(cube_path)
    mask_np = load_mask(label_dir / f"{stem}.png")
    present = [c for c in range(len(CLASS_NAMES)) if (mask_np == c).any()]
    counts = class_counts(mask_np)
    print(f"stem {stem}: {[CLASS_NAMES[c] for c in present]}")

    rgb = cube_np[list(FALSE_COLOUR_BANDS)].transpose(1, 2, 0).copy()
    for k in range(3):
        lo, hi = np.percentile(rgb[..., k], [2, 98])
        rgb[..., k] = np.clip((rgb[..., k] - lo) / (hi - lo + 1e-6), 0, 1)
    colour = np.zeros((*mask_np.shape, 3))
    for c, name in enumerate(CLASS_NAMES):
        colour[mask_np == c] = COLOR[name]

    fig, axes = plt.subplots(
        1, 3, figsize=(14.8, 5.55), gridspec_kw={"width_ratios": [1, 1, 1.45]}, constrained_layout=True
    )
    nm = ", ".join(f"{v:.0f}" for v in nominal_nm(FALSE_COLOUR_BANDS))
    axes[0].imshow(rgb)
    axes[0].set_title(f"false colour, bands {FALSE_COLOUR_BANDS}\napprox. {nm} nm, infrared, display stretch", fontsize=11)
    axes[0].axis("off")
    axes[1].imshow(colour)
    axes[1].set_title(
        f"label mask, labels_hyper_lt\nfilament is the thin green curve, {int(counts[5]):,} px",
        fontsize=11,
    )
    axes[1].axis("off")
    axes[1].legend(
        handles=[Patch(facecolor=COLOR[n], edgecolor="0.35", label=f"{i} {LABEL[n]}") for i, n in enumerate(CLASS_NAMES)],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=4,
        frameon=False,
        fontsize=9,
    )

    bands = np.arange(N_BANDS)
    for c in present:
        name = CLASS_NAMES[c]
        spectrum = cube_np[:, mask_np == c].mean(axis=1)
        draw_line(axes[2], bands, spectrum, name, label=f"{LABEL[name]}  {counts[c]:,} px")
    axes[2].set_xlim(0, N_BANDS - 1)
    axes[2].set_ylim(0, 1)
    axes[2].set_xlabel("band index")
    axes[2].set_ylabel("scaled value  (uint16 / 65535)")
    axes[2].secondary_xaxis("top", functions=(nominal_nm, band_of_nm)).set_xlabel(
        "nominal nm, even spacing assumed", fontsize=10
    )
    axes[2].grid(color="#f0f0f0")
    axes[2].set_axisbelow(True)
    axes[2].legend(frameon=False, fontsize=9, loc="upper left", borderaxespad=0.2)
    axes[2].set_title("mean spectrum of pixels in this cube\nscaled values, not display-stretched", fontsize=11)
    named = ", ".join(LABEL[CLASS_NAMES[c]] for c in present if c)
    fig.suptitle(f"One train cube, {stem}    ·    {named}", fontsize=14, color=INK, ha="left", x=0.0)
    save(fig, "data_example")


## D. Mean spectrum per class (slide 3 or 6)

Needs `SW_ROOT`. For each object class, take the 8 train cubes with the most pixels of that class (every train mask is scanned; only those cubes are read). The line is the mean of the 8 per-cube mean spectra. The shaded band is cardboard only, ±1 standard deviation across its 8 cubes. Shading every class hid the lines. Background is the background pixels in that same set of cubes.

The legend is the share of that class's train pixels inside its 8 cubes. The right panel is a different population: the class share of every train pixel, from `sw.data`.


In [ ]:
if not SW_ROOT:
    print("SW_ROOT is not set; skipping cell D.")
else:
    label_dir, cube_dir = train_dirs()
    chosen = choose_class_cubes(label_dir, PER_CLASS)
    union = list(dict.fromkeys(stem for stems in chosen.values() for stem in stems))
    missing = [stem for stem in union if not (cube_dir / f"{stem}.tiff").is_file()]
    if missing:
        raise FileNotFoundError(f"{len(missing)} cubes missing under {cube_dir}, first {missing[0]}")
    print(f"{len(union)} cubes for {PER_CLASS} richest masks of each object class")

    cube_means = {name: [] for name in (*chosen, "background")}
    pixels = {name: 0 for name in cube_means}
    for stem in union:
        cube = load_cube(cube_dir / f"{stem}.tiff")
        mask = load_mask(label_dir / f"{stem}.png")
        names = {name for name, stems in chosen.items() if stem in stems}
        names.add("background")
        for name in names:
            sel = mask == CLASS_NAMES.index(name)
            if not np.any(sel):
                continue
            pixels[name] += int(sel.sum())
            cube_means[name].append(cube[:, sel].mean(axis=1))

    fig, (ax, ax2) = plt.subplots(
        1, 2, figsize=(14.5, 6.9), gridspec_kw={"width_ratios": [3.15, 1]}
    )
    bands = np.arange(N_BANDS)
    for g in range(N_GROUPS):
        if g % 2 == 0:
            ax.axvspan(g * GROUP - 0.5, (g + 1) * GROUP - 0.5, color="black", alpha=0.035, linewidth=0, zorder=0)
    for b in FALSE_COLOUR_BANDS:
        ax.axvline(b, color="#9a9a9a", lw=0.7, ls=":", zorder=1)

    draw_order = ["background", *OBJECT_CLASSES]
    cardboard_note = None
    for name in draw_order:
        arr = np.stack(cube_means[name])
        mean, std = arr.mean(axis=0), arr.std(axis=0)
        # One shade only. Shading every class turned the middle of the chart into one beige band.
        if name == "cardboard":
            ax.fill_between(bands, mean - std, mean + std, color=COLOR[name], alpha=0.18, linewidth=0, zorder=1)
            peak = int(np.argmax(mean))
            cardboard_note = (bands[peak], min(0.97, float(mean[peak] + std[peak]) + 0.03))
        share = pixels[name] / HYPER_LT_TRAIN_PIXEL_COUNTS[CLASS_NAMES.index(name)]
        if name == "background":
            label = f"background · {len(arr)} cubes"
        else:
            label = f"{LABEL[name]} · {share:.0%} of its pixels"
        lw = 2.6 if name == "filament" else 2.2
        draw_line(ax, bands, mean, name, label=label, lw=lw)
    if cardboard_note:
        ax.text(
            cardboard_note[0],
            cardboard_note[1],
            "cardboard",
            ha="center",
            va="bottom",
            fontsize=11,
            color=INK,
            zorder=5,
            bbox=dict(facecolor="white", edgecolor="none", pad=1.2, alpha=0.9),
        )
    ax.set_xlim(0, N_BANDS - 1)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel(
        "band index    stripes: 14 token groups of 16    dotted: false-colour bands\n"
        "top axis is nominal nm, even spacing 900 to 1700; meta.json has no wavelengths"
    )
    ax.set_ylabel("scaled value, mean of cube means")
    top = ax.secondary_xaxis("top", functions=(nominal_nm, band_of_nm))
    top.tick_params(labelsize=9, colors=MUTED)
    ax.grid(color="#f2f2f2")
    ax.set_axisbelow(True)
    ax.set_title(
        "Cardboard separates. Film, bag, and filament overlap the background.\n"
        f"Each object line: the {PER_CLASS} train cubes with the most pixels of that class. Shade: cardboard ±1 sd.",
        loc="left",
        fontsize=11,
        color=INK,
        pad=8,
    )
    ax.legend(
        frameon=False,
        ncol=4,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.22),
        fontsize=9,
    )

    share = np.array(HYPER_LT_TRAIN_PIXEL_COUNTS) / sum(HYPER_LT_TRAIN_PIXEL_COUNTS) * 100
    ys = np.arange(len(CLASS_NAMES))[::-1]
    ax2.barh(ys, share, color=[COLOR[n] for n in CLASS_NAMES], edgecolor=INK, linewidth=0.7, height=0.72)
    ax2.set_yticks(ys, [LABEL[n] for n in CLASS_NAMES])
    ax2.set_xscale("log")
    ax2.set_xlim(0.04, 200)
    ax2.set_xlabel("% of train pixels")
    for yi, s in zip(ys, share):
        ax2.text(s * 1.15, yi, f"{s:.3g}%", va="center", fontsize=9, color=INK)
    ax2.set_title("Full train split\nnot this 8-cube sample", loc="left", fontsize=12)
    fig.subplots_adjust(left=0.07, right=0.985, top=0.86, bottom=0.20, wspace=0.34)
    save(fig, "band_signatures")
    for name in draw_order:
        total = HYPER_LT_TRAIN_PIXEL_COUNTS[CLASS_NAMES.index(name)]
        print(f"{LABEL[name]:12} cubes {len(cube_means[name]):2}  pixels {pixels[name]:8}  share {pixels[name] / total:.1%}")


## G. Training curves from `metrics.csv` (Nov 5)

The harness writes `epoch, train_loss, val_fg_miou, steps, iou_<class>` per epoch. Copy the CNN run's file to `docs/metrics/cnn_ds2.csv` (a few KB; `docs/runs/` would be caught by the `runs/` ignore rule) and point the transformer entry at its run folder. Entries whose file is missing are skipped.


In [ ]:
import csv

RUNS = {
    "CNN control, cnn_ds2": REPO / "docs" / "metrics" / "cnn_ds2.csv",
    "band-token transformer": REPO / "runs" / "transformer" / "metrics.csv",
}
found = {name: path for name, path in RUNS.items() if path.is_file()}
if not found:
    print("No metrics file found. Copy cnn_ds2/metrics.csv to docs/metrics/cnn_ds2.csv, or edit RUNS.")
else:
    fig, (ax_l, ax_m) = plt.subplots(1, 2, figsize=(12.2, 4.8), constrained_layout=True)
    styles = [("-", INK), ("--", "#555555"), (":", MUTED)]
    for (name, path), (ls, color) in zip(found.items(), styles):
        with path.open(newline="") as handle:
            rows = list(csv.DictReader(handle))
        epochs = np.array([int(r["epoch"]) for r in rows])
        loss = np.array([float(r["train_loss"]) for r in rows])
        miou = np.array([float(r["val_fg_miou"]) * 100 if r["val_fg_miou"] else np.nan for r in rows])
        ax_l.plot(epochs, loss, ls, color=color, lw=2.2, label=name)
        ax_m.plot(epochs, miou, ls, color=color, lw=2.2, label=name)
        best = int(np.nanargmax(miou))
        ax_m.scatter([epochs[best]], [miou[best]], s=42, color=color, zorder=3)
        ax_m.annotate(
            f"{miou[best]:.1f}  epoch {epochs[best]}",
            xy=(epochs[best], miou[best]),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            fontsize=10,
            color=INK,
        )
    ax_l.set_xlabel("epoch")
    ax_l.set_ylabel("train loss")
    ax_l.set_title("Train loss", loc="left")
    ax_l.grid(color="#f0f0f0")
    ax_l.legend(frameon=False, loc="upper right")
    ax_m.set_xlabel("epoch")
    ax_m.set_ylabel("val FG mIoU (%)")
    ax_m.set_title("Val foreground mIoU", loc="left")
    ax_m.grid(color="#f0f0f0")
    ax_m.set_ylim(0, 70)
    ax_m.axhline(52.8, color=MUTED, ls=":", lw=1)
    ax_m.axhline(54.3, color="#b0b0b0", ls=":", lw=1)
    ax_m.annotate(
        "paper MiniNet-v2 52.8\nSegFormer-B0 54.3\nboth full res, 200 epochs",
        xy=(0.98, 54.3),
        xycoords=("axes fraction", "data"),
        ha="right",
        va="bottom",
        fontsize=9,
        color=MUTED,
    )
    ax_m.legend(frameon=False, loc="lower right")
    save(fig, "training_curves")
